# IC-1005 — Case study: is this portfolio really two bets?

**ML & FinTech · 115-1 · 20261005 · Session 3 · work in pairs · 40 minutes, then discussion**

| Step | Min | What you do |
|---|---|---|
| 0 | 3 | Sit with a partner. Each of you gets a different client |
| 1 | 7 | PCA on six funds you can see |
| 2 | 15 | Test the manager's claim, then compare with your partner |
| 3 | 10 | What is PC1? Would your partner's stock help your client? |
| 4 | 5 | Hand in **one sheet of paper per pair** |

---

## The case

You work at a robo-advisor. Every analyst has a different client, and every client holds ten
stocks. Your client's portfolio manager has sent this report:

> *"We ran a principal components analysis on your price history. **Two components explain
> most of your portfolio.** It is essentially two bets."*

Your client asks: **"Is that right?"**

Your stocks are drawn from your 學號, so **your numbers are yours alone**. Your partner has a
different client and will get different numbers. You need each other's results in Steps 2 and 3.

You may use any AI tool for the code. The graded part is the paper your pair hands in at the
end, and what you say when called on.

In [ ]:
import hashlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

---
# Step 0 — Your client (3 min)

Fill in both IDs, then run the next two cells.

In [ ]:
STUDENT_ID = ""    # your 學號, in quotes, e.g. "315707999"
PARTNER_ID = ""    # your partner's 學號

In [ ]:
URL  = ("https://raw.githubusercontent.com/202609-ML-FinTech/00-course-info/"
        "main/in-class-exercise/historical_stock_prices_2017.csv")
CAND = ["historical_stock_prices_2017.csv",
        "../historical_stock_prices_2017.csv",
        "../00-course-info/in-class-exercise/historical_stock_prices_2017.csv"]
path = next((p for p in CAND if Path(p).exists()), URL)   # local file if you have it, else GitHub
prices = pd.read_csv(path, index_col=0).dropna(axis=1)    # 50 stocks, 251 trading days of 2017
prices.index = pd.to_datetime(prices.index)

def draw_portfolio(student_id, k=10):
    """The same 學號 always gives the same k stocks; different 學號, different stocks."""
    key = lambda t: hashlib.sha256(f"{int(student_id)}-{t}".encode()).hexdigest()
    return sorted(sorted(prices.columns, key=key)[:k])

assert str(STUDENT_ID).strip().isdigit(), "Fill in STUDENT_ID in the cell above, then run again."
MY_PORTFOLIO = draw_portfolio(STUDENT_ID)
P = prices[MY_PORTFOLIO]           # your client's 10 stocks: daily prices
R = np.log(P).diff().dropna()      # daily log returns
print("Client of", STUDENT_ID, "holds:", MY_PORTFOLIO)
print(P.shape, R.shape)

---
# Step 1 — PCA on something you can see (7 min)

Six funds, two numbers each. The code is complete: run it.

In [ ]:
funds = pd.DataFrame({
    "fund": list("ABCDEF"),
    "ret":  [2.0, 4.0, 6.0,  8.0, 10.0, 12.0],   # annual return, %
    "risk": [3.0, 5.0, 8.0, 10.0, 13.0, 15.0],   # annual volatility, %
}).set_index("fund")

Z = StandardScaler().fit_transform(funds)        # centre and scale
pca = PCA().fit(Z)
print("PVE:", (pca.explained_variance_ratio_ * 100).round(1))
print("PC1 loadings:", pca.components_[0].round(3))

plt.scatter(funds["ret"], funds["risk"], s=80)
for n, r in funds.iterrows():
    plt.annotate(n, (r["ret"], r["risk"]))
plt.xlabel("return (%)"); plt.ylabel("risk (%)"); plt.title("Six funds")
plt.show()

**Q1.** PC1 explains almost everything, and its two loadings have the **same sign**.
In one sentence: what does a fund with a high PC1 score look like?

> _replace this line_

*The sign of a whole component is arbitrary: `[0.707, 0.707]` and `[-0.707, -0.707]` mean the
same thing. Read loadings relative to each other.*

---
# Step 2 — Test the manager's claim (15 min)

## 2a. Do what the manager did (5 min)

He ran PCA on the **prices** `P`, standardised, with the ten stocks as variables.
Print the PVE of each component and the cumulative PVE.

*Reminder: `StandardScaler().fit_transform(...)`, `PCA().fit(...)`,
`.explained_variance_ratio_`, `np.cumsum(...)`.*

In [ ]:
# 2a: PCA on prices



## 2b. Now on returns (5 min)

Run the same PCA on the daily returns `R`.
Print the PVE, the cumulative PVE, and **how many components you need to reach 80%**.

In [ ]:
# 2b: PCA on returns



**Q2.** Your numbers:

| | PC1 + PC2 (%) | components needed for 80% |
|---|---|---|
| prices | | |
| returns | | |

**Q3.** The two runs disagree. Which one answers the client's question about his **risk**, and
why is the other one misleading? Think about what a price does over a year that a return does not.

> _replace this line_

## 2c. Compare with your partner (5 min)

**Q4.** Write down your partner's two "PC1 + PC2" numbers.

| | prices | returns |
|---|---|---|
| me | | |
| my partner | | |

You both ran the same code. Give **one reason** the numbers differ, and say whose portfolio looks
more like "a few bets" on the returns measure.

> _replace this line_

---
# Step 3 — What is PC1? (10 min)

Use the **returns** PCA from 2b.

## 3a. Is PC1 your client's market factor? (5 min)

1. Print the PC1 loading of each of your ten stocks.
2. Compute the PC1 **score** of each day, `pca.transform(...)[:, 0]`, and its correlation with
   your equal-weighted portfolio return, `R.mean(axis=1)`. Ignore the sign.

In [ ]:
# 3a: PC1 loadings, and what PC1 tracks



**Q5.** My correlation is: ____

Rule of thumb: **0.8 or more** means PC1 is the portfolio's "market", the days everything moves
together. Below that, PC1 is something narrower. Which is it for your client? If it is narrower,
which stocks have the largest loadings?

> _replace this line_

**Q6.** Which of your stocks has the PC1 loading **closest to zero**? That stock moves least with
PC1.

> Ticker: ____ , loading: ____

## 3b. Would your partner's stock help your client? (5 min)

Ask your partner for their answer to Q6. Then measure how that stock moves with **your** portfolio.

In [ ]:
partner_stock = ""     # your partner's ticker from Q6, in quotes

ew = R.mean(axis=1)                                   # your equal-weighted portfolio return
r_partner = np.log(prices[partner_stock]).diff().dropna()
print(partner_stock, "vs my portfolio:", round(r_partner.corr(ew), 2))

**Q7.** The correlation is: ____

Would adding this stock diversify your client? Rule of thumb: **below 0.3** it would.
(If you already hold that stock, say so.)

> _replace this line_

---
# Step 4 — Hand in, on paper (5 min)

**One sheet per pair.** Write:

1. Both names and both 學號.
2. For **each** of you: PC1 + PC2 on prices, PC1 + PC2 on returns, and the Q5 correlation.
3. **Two sentences, agreed between you:**
   - Is the manager right that these portfolios are "two bets"? Why or why not?
   - Is PC1 always "the market"? Use your two correlations.

Give the sheet to Prof. Teng or Claire before you leave. **The sheet is the graded part.**

Then: *Restart & Run All*, and push this notebook as `in-class-exercise/1005/IC-1005.ipynb`
in your own repository before **12:10**.